In [1]:
%pip install matplotlib seaborn pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


___
___
# FASE 2: Deteccion de Fugas en Margenes Unitarios
___
___

___
## 1. Librerias
___


* Importar librerias.

In [2]:
import pandas as pd
import numpy as np

___
## 2. Data
___

### 2.1. Cargar la data sintetica.

In [3]:
df_compras = pd.read_csv('data/compras_proveedores_full.csv')
df_ventas = pd.read_csv('data/ventas_diarias_full.csv')

### 2.2. Convertir fechas a formato datetime para poder operar cronológicamente

In [4]:
df_compras['fecha'] = pd.to_datetime(df_compras['fecha'])
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

### 2.3. Ordenar compras cronológicamente para asegurar que busquemos el costo correcto

In [5]:
df_compras = df_compras.sort_values('fecha')

___
## 3. Funcion Sabueso
___

### 3.1. Encontrar el costo de reposición real a la fecha de la venta.

In [6]:
def obtener_costo_reposicion(row):
    # Buscamos todas las compras de ESTE producto ocurridas ANTES o el MISMO DÍA de la venta
    compras_previas = df_compras[
        (df_compras['producto_id'] == row['producto_id']) & 
        (df_compras['fecha'] <= row['fecha'])
    ]
    
    if not compras_previas.empty:
        # Tomamos el costo de la compra más reciente (última fila del subconjunto)
        return compras_previas.iloc[-1]['costo_unitario']
    else:
        # Si por alguna falla no hay compra previa registrada, usamos el costo simulado como salvaguarda
        return row['costo_asociado_simulado']



### 3.2. Aplicamos el sabueso línea por línea en las ventas

In [7]:
print("Analizando transacciones y cruzando costos de reposición...")
df_ventas['costo_reposicion'] = df_ventas.apply(obtener_costo_reposicion, axis=1)

Analizando transacciones y cruzando costos de reposición...


___
## 4. Calculo de Metricas Criticas.
___

### 4.1. Margen Unitario en Dólares = Precio Venta - Costo Reposición

In [8]:
df_ventas['margen_unitario_usd'] = df_ventas['precio_unitario'] - df_ventas['costo_reposicion']

### 4.2. Porcentaje de Margen Bruto = (Margen Unitario / Precio Venta) * 100

In [9]:
df_ventas['porcentaje_margen'] = (df_ventas['margen_unitario_usd'] / df_ventas['precio_unitario']) * 100

### 4.3. Fuga Total en la Transacción = Margen Unitario Negativo * Cantidad Vendida

⚠️ Si el margen es positivo, la fuga es 0.

In [10]:
df_ventas['fuga_transaccion_usd'] = np.where(
    df_ventas['margen_unitario_usd'] < 0, 
    abs(df_ventas['margen_unitario_usd']) * df_ventas['cantidad_vendida'], 
    0
)

___
## 5. Extraccion de resultados para el cliente
___

### 5.1. Fuga total detectada.

In [11]:
fugas_detectadas = df_ventas[df_ventas['fuga_transaccion_usd'] > 0]
fuga_total_negocio = df_ventas['fuga_transaccion_usd'].sum()
total_ventas_periodo = (df_ventas['precio_unitario'] * df_ventas['cantidad_vendida']).sum()

print("\n" + "="*50)
print("     ¡REPORTE DEL SABUESO: FUGAS DE MARGEN!")
print("="*50)
print(f"Total facturado en el período: ${total_ventas_periodo:,.2f} USD")
print(f"Fuga total detectada (Dinero perdido): ${fuga_total_negocio:,.2f} USD")
print(f"Porcentaje de ingresos destruidos por fugas: {(fuga_total_negocio/total_ventas_periodo)*100:.2f}%")
print(f"Número de transacciones en rojo: {len(fugas_detectadas)} de {len(df_ventas)}")
print("="*50)



     ¡REPORTE DEL SABUESO: FUGAS DE MARGEN!
Total facturado en el período: $1,952,099.07 USD
Fuga total detectada (Dinero perdido): $9,088.29 USD
Porcentaje de ingresos destruidos por fugas: 0.47%
Número de transacciones en rojo: 529 de 6938


### 5.2. Epicentro de la fuga.

* Ver los 5 peores productos.

In [12]:
top_fugas_prod = df_ventas.groupby('producto_id').agg(
    fuga_total=('fuga_transaccion_usd', 'sum'),
    cantidad_afectada=('cantidad_vendida', lambda x: df_ventas.loc[x.index, 'cantidad_vendida'].where(df_ventas.loc[x.index, 'margen_unitario_usd'] < 0).sum())
).sort_values(by='fuga_total', ascending=False)

print("\nEpicentro de la Fuga por Producto:")
print(top_fugas_prod.head(3))


Epicentro de la Fuga por Producto:
             fuga_total  cantidad_afectada
producto_id                               
PROD016         3193.25             2285.0
PROD025         2718.25              386.0
PROD007         2186.63              488.0


___
## 6. Visualizacion de Resultados
___

### 6.1. Importar librerias de visualizacion.

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns

### 6.2. Configuración de estilo visual.

In [14]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

### 6.3. Margen de Utilidad Producto Estrella.

* Agrupamos por mes para ver la tendencia de las ventas vs la degradación del margen.

In [16]:
df_ventas['mes_nombre'] = df_ventas['fecha'].dt.strftime('%B')
df_prod_estrella = df_ventas[df_ventas['producto_id'] == 'PROD002']

analisis_mensual = df_prod_estrella.groupby(df_ventas['fecha'].dt.to_period('M')).agg(
    volumen_vendido=('cantidad_vendida', 'sum'),
    margen_promedio_porcentaje=('porcentaje_margen', 'mean'),
    ingreso_total=('precio_unitario', lambda x: (x * df_prod_estrella.loc[x.index, 'cantidad_vendida']).sum())
).reset_index()

analisis_mensual['fecha'] = analisis_mensual['fecha'].dt.to_timestamp()

* Crear el gráfico de doble eje.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

# Eje 1: Volumen de Ventas (Barras)
color = '#A9A9A9' # Gris 
ax1.set_xlabel('Período (Meses 2026)', fontsize=12, fontweight='bold', labelpad=12)
ax1.set_ylabel('Volumen Vendido (Unidades)', color='#666666', fontsize=11)
barras = ax1.bar(analisis_mensual['fecha'].dt.strftime('%b'), analisis_mensual['volumen_vendido'], color=color, alpha=0.6, width=0.4, label='Unidades Vendidas')
ax1.tick_params(axis='y', labelcolor='#666666')
ax1.grid(False) # Limpiar ruido visual

# Eje 2: Margen Real % (Línea Roja - La Fuga Silenciosa)
ax2 = ax1.twinx()
color_linea = '#D32F2F' # Rojo de alerta financiero
ax2.set_ylabel('Margen Bruto Real (%)', color=color_linea, fontsize=11, fontweight='bold')
linea = ax2.plot(analisis_mensual['fecha'].dt.strftime('%b'), analisis_mensual['margen_promedio_porcentaje'], color=color_linea, marker='o', linewidth=3, markersize=8, label='Margen Bruto %')
ax2.tick_params(axis='y', labelcolor=color_linea)

# Línea de peligro (Margen 0% o umbral mínimo deseado, ej: 20%)
ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)

plt.title('El Espejismo Comercial: Más Ventas, Menos Ganancia\n(Caso: Balde de Aceite Diesel 15W-40)', fontsize=14, fontweight='bold', pad=20, color='#1A1A1A')
fig.tight_layout()
plt.savefig('figures/portada_linea_temporal.png', dpi=300)
plt.close()

### 6.4. Analisis Tactico

* Ranking de Fugas por Vendedor

In [18]:
ranking_vendedores = df_ventas.groupby('vendedor').agg(
    total_facturado=('precio_unitario', lambda x: (x * df_ventas.loc[x.index, 'cantidad_vendida']).sum()),
    dinero_perdido_fuga=('fuga_transaccion_usd', 'sum'),
    transacciones_en_rojo=('fuga_transaccion_usd', lambda x: (x > 0).sum())
).sort_values(by='dinero_perdido_fuga', ascending=False).reset_index()

* Gráfico de Barras

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(
    x='dinero_perdido_fuga', 
    y='vendedor', 
    data=ranking_vendedores, 
    palette='Reds_r',
    hue='vendedor',  # <--- Asignamos la variable categórica al color
    legend=False     # <--- Ocultamos la leyenda automática que genera 'hue'
)

# Añadir etiquetas con los montos exactos en las barras
for index, value in enumerate(ranking_vendedores['dinero_perdido_fuga']):
    plt.text(value, index, f'  ${value:,.2f}', va='center', fontweight='bold', color='#2C3E50')

plt.tight_layout()
plt.savefig('figures/ranking_vendedores.png', dpi=300)
plt.close()

print("¡Gráficos de la propuesta comercial generados con éxito!")

¡Gráficos de la propuesta comercial generados con éxito!
